In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader
from src.data.dataset import FundusDataset
from transformers import ViTForImageClassification
from transformers import ViTForImageClassification, ViTImageProcessor

def cpu():
    if torch.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

result = cpu()
print(result)
device = torch.device(result)
print(device)

train_dir = r'F:\graduation_project\data\processed\Training Images'
train_excel_dir = r'F:\graduation_project\data\processed\training annotation (English).xlsx'

# 训练集
train_dataset = FundusDataset(train_dir , train_excel_dir , is_training=True)

# 创建DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

# 用已经预训练过的模型可以增加准确率
local_model_path = r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224'


# 加载预训练的处理器
processor = ViTImageProcessor.from_pretrained(local_model_path)

# 加载预训练的ViT模型
model = ViTForImageClassification.from_pretrained(
    local_model_path,
    num_labels=8,  # 指定分类数
    ignore_mismatched_sizes=True  # 允许模型权重尺寸不匹配时自动调整
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

# 优化器
optimizer = torch.optim.AdamW(  #AdamW收敛快，准确率高，泛化性好
    model.parameters(),
    lr = 0.01,
)

for epoch in range(20):
    model.train()
    train_loss = 0
    
    for image, labels, img_name in train_loader:
        
        image = image.to(device)
        labels = labels.to(device)
    
        optimizer.zero_grad() # 清空之前的梯度
        outputs = model(image)  
        loss = criterion(outputs.logits, labels)  # outputs中包含了多个属性，需要转化成张量

        loss.backward()  # 反向传播(计算梯度)
        optimizer.step() # 利用梯度来更新参数，优化

        train_loss += loss
        
    print(train_loss)

cuda
cuda


Some weights of ViTForImageClassification were not initialized from the model checkpoint at C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 